# EPI: Empirical Likelihood Meets Prediction-Powered Inference

Independent implementation of Wang, Wen & Zou (2025), arXiv:2512.16363v1.
Uses the same `PPIData -> target -> inference -> InferenceResult` flow as `ppi.ipynb`.
The base comparison uses predictions alone for every method. An additional example
adds covariates, as in the paper's Section 4.1. This notebook is a small demonstration,
not a reproduction of the paper's 10,000-replication experiments.


In [1]:
import numpy as np
import pandas as pd
from openppi import EPI, Mean, LinearRegression, PPIData, RegressionData, get_method


def make_data(seed, n=200, m=2000, distribution='normal'):
    rng = np.random.default_rng(seed)
    draw = (lambda count: rng.normal(size=count)) if distribution == 'normal' else (lambda count: rng.exponential(scale=.5, size=count))
    x, xu = draw(n), draw(m)
    y = x + draw(n)
    f, fu = x + rng.normal(size=n), xu + rng.normal(size=m)
    return PPIData(y, f, fu), x, xu


data, x, xu = make_data(42)
target = Mean(outcome_type='continuous')
methods = {name: get_method(name).inference for name in ('PPI', 'PPI++', 'EPI', 'EPI-CF')}


## 1. Estimate the mean with the existing PPI interface

EPI uses positive weights on labeled outcomes, constrained to match the pooled
auxiliary mean. Its default mean confidence interval inverts the empirical
likelihood ratio with the finite-unlabeled-sample correction in Theorem 2.
PPI and PPI++ return Wald intervals. `standard_error` for EPI uses Theorem 1.


In [2]:
rows = []
for name, estimator in methods.items():
    result = estimator(data, target=target)
    lo, hi = result.confidence_interval
    rows.append(dict(method=name, estimate=result.estimate, se=result.standard_error,
                     lower=lo, upper=hi, length=hi-lo))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

result = EPI()(data, target=target)
print('Weight sum:', result.diagnostics['weights'].sum())
print('Maximum auxiliary residual:', np.max(np.abs(result.diagnostics['moment_residual'])))
print('Finite-m LR scale:', result.diagnostics['el_ratio_scale'])


method  estimate     se   lower  upper  length
   PPI   -0.0852 0.1055 -0.2920 0.1216  0.4137
 PPI++   -0.0304 0.0876 -0.2022 0.1413  0.3435
   EPI   -0.0331 0.0875 -0.2053 0.1393  0.3446
EPI-CF   -0.0348 0.0883 -0.2087 0.1391  0.3478
Weight sum: 1.0
Maximum auxiliary residual: 6.889282867814788e-18
Finite-m LR scale: 1.025026314656609


## 2. Enrich the auxiliary function with covariates

For Section 4.1's auxiliary design `h=(prediction, X)`, use `RegressionData`
even when the target is a mean. Labels and both samples remain independent;
all predictors must be trained independently of these inference labels.
Increasing `degree` adds polynomial terms and interactions. Excessive basis
dimension or a pooled auxiliary mean outside the labeled convex hull raises
an explicit error rather than substituting another method.


In [3]:
with_covariates = RegressionData(data.labels, data.labeled_predictions,
                                 data.unlabeled_predictions, x[:, None], xu[:, None])
for estimator in (EPI(), EPI(auxiliary='crossfit')):
    result = estimator(with_covariates, target=target)
    print(estimator.name, 'estimate:', round(result.estimate, 4),
          'SE:', round(result.standard_error, 4), 'CI:', result.confidence_interval)


EPI estimate: -0.0195 SE: 0.0778 CI: (-0.17419859429533707, 0.13204950386640674)
EPI-CF estimate: -0.0215 SE: 0.0781 CI: (-0.17638084109854466, 0.13041935611629268)


## 3. Linear regression

The same EL weights solve the weighted labeled least-squares score. Regression
returns the full coefficient covariance and coordinatewise Wald intervals.
The joint weighted-chi-square EL confidence region from the paper is outside
this implementation. `degree=2` includes the score-relevant pairwise products.


In [4]:
rng = np.random.default_rng(8)
xl, xu = rng.normal(size=(400, 2)), rng.normal(size=(2000, 2))
y = 1 + xl @ np.array([2., -1.]) + rng.normal(size=400)
fl = 1 + xl @ np.array([1.5, -.8])
fu = 1 + xu @ np.array([1.5, -.8])
regression = RegressionData(y, fl, fu, xl, xu)
result = EPI(degree=2)(regression, target=LinearRegression())
print('Coefficients [intercept, slope 1, slope 2]:', result.estimate)
print('Standard errors:', result.standard_error)
print('Marginal 95% intervals:', result.confidence_interval)


Coefficients [intercept, slope 1, slope 2]: [ 1.05098863  2.0095629  -1.01644891]
Standard errors: [0.04779095 0.04800674 0.04744498]
Marginal 95% intervals: (array([ 0.95732009,  1.91547142, -1.10943937]), array([ 1.14465718,  2.10365439, -0.92345846]))


## 4. Small coverage experiment: normal and skewed outcomes

Report RMSE, coverage, and average interval length over 100 repetitions per
scenario. The truth is 0 for the normal design and 1 for the exponential design
(two independent exponential variables each with mean 0.5). Monte Carlo
coverage fluctuates at this scale; increase `repetitions` for precise comparisons.
Cross-fitting uses equal folds as in equation (5): both sample counts must be
divisible by `folds`. The default learner is NumPy least squares; a fresh
`fit/predict` regressor can be supplied with `learner_factory`.


In [5]:
repetitions = 100
summary = []
for distribution, truth in [('normal', 0.), ('exponential', 1.)]:
    records = {name: [] for name in methods}
    for rep in range(repetitions):
        sample, _, _ = make_data(1000 + rep, n=100, m=1000, distribution=distribution)
        for name, estimator in methods.items():
            result = estimator(sample, target=target)
            lo, hi = result.confidence_interval
            records[name].append(((result.estimate - truth)**2, lo <= truth <= hi, hi-lo))
    for name, values in records.items():
        mse, coverage, length = np.mean(values, axis=0)
        summary.append(dict(distribution=distribution, method=name, rmse=np.sqrt(mse),
                            coverage=coverage, average_length=length))
print(pd.DataFrame(summary).to_string(index=False, float_format=lambda v: f'{v:.4f}'))


distribution method   rmse  coverage  average_length
      normal    PPI 0.1664    0.9200          0.5810
      normal  PPI++ 0.1384    0.9200          0.4837
      normal    EPI 0.1382    0.9100          0.4874
      normal EPI-CF 0.1388    0.9200          0.4916
 exponential    PPI 0.1036    0.9600          0.4577
 exponential  PPI++ 0.0672    0.9300          0.2565
 exponential    EPI 0.0675    0.9300          0.2581
 exponential EPI-CF 0.0688    0.9300          0.2629


## Scope

Implemented: pooled-centering basis EPI (equation 4), equal-fold cross-fitted EPI
(equation 5), mean profile EL intervals with the scalar Theorem 2 correction,
and just-identified mean/regression covariance from Theorems 1 and 3.
Not included: over-identified supervised scores, distribution-functional
confidence sets, known-population auxiliary means, or joint regression EL regions.
See `docs/methods.md` and `tests/test_epi.py` for formulas and numerical checks.
